In [0]:
# Import libraries
import requests
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType

spark = SparkSession.builder.getOrCreate()

# 1️⃣ Call API
url = "https://jsonplaceholder.typicode.com/users"
response = requests.get(url)
data = response.json()  # List of dictionaries

# 2️⃣ Define nested schema
geo_schema = StructType([
    StructField("lat", StringType(), True),
    StructField("lng", StringType(), True)
])

address_schema = StructType([
    StructField("street", StringType(), True),
    StructField("suite", StringType(), True),
    StructField("city", StringType(), True),
    StructField("zipcode", StringType(), True),
    StructField("geo", geo_schema, True)
])

company_schema = StructType([
    StructField("name", StringType(), True),
    StructField("catchPhrase", StringType(), True),
    StructField("bs", StringType(), True)
])

user_schema = StructType([
    StructField("id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("username", StringType(), True),
    StructField("email", StringType(), True),
    StructField("address", address_schema, True),
    StructField("phone", StringType(), True),
    StructField("website", StringType(), True),
    StructField("company", company_schema, True)
])

# 3️⃣ Convert API JSON directly to Spark DataFrame
df_spark = spark.createDataFrame(data, schema=user_schema)

# 4️⃣ Flatten nested columns for Silver/Gold layer
df_flat = df_spark.select(
    col("id"),
    col("name"),
    col("username"),
    col("email"),
    col("address.street").alias("street"),
    col("address.suite").alias("suite"),
    col("address.city").alias("city"),
    col("address.zipcode").alias("zipcode"),
    col("address.geo.lat").alias("lat"),
    col("address.geo.lng").alias("lng"),
    col("phone"),
    col("website"),
    col("company.name").alias("company_name"),
    col("company.catchPhrase").alias("company_catchphrase"),
    col("company.bs").alias("company_bs")
)

# 5️⃣ Show final flattened DataFrame
# df_flat.show(truncate=False)
df_flat.printSchema()

In [0]:
# Write DataFrame to Delta format with partitioning
df_flat.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("city") \
    .save("/Volumes/banking_api/default/users_banking_volume/users")

In [0]:
import requests
import random
from datetime import datetime, timedelta

data = requests.get("https://jsonplaceholder.typicode.com/todos").json()

transactions = []
for row in data[:50]:
    txn = {
        "transaction_id": row["id"],
        "account_id": row["userId"],
        "amount": random.randint(100, 10000),   # ✅ generated
        "type": random.choice(["DEBIT", "CREDIT"]),
        "status": "SUCCESS" if row["completed"] else "PENDING",
        "timestamp": (datetime.now() - timedelta(days=random.randint(0,5))).isoformat()
    }
    transactions.append(txn)

df=spark.createDataFrame(transactions)
df.write.mode("overwrite").format("delta").save("/Volumes/banking_api/default/users_banking_volume/transactions")